# LLM API 與 Ollama——超越 OpenAI

_重要：若你對 API 或 PC/Mac 上的環境變數不熟悉，請先閱讀 Guide 4 技術基礎中的 API 章節（Guide 4 主題 3 與 5），再繼續本指南。_

## 使用 OpenAI 以外模型的關鍵背景——請先讀！

整門課程我們用 API 連接世上最強的 LLM。

OpenAI、Anthropic、Google、DeepSeek 等公司建立了 web endpoint。你對 web 位址發 HTTP 請求，並傳入 prompt 相關資訊來呼叫模型。

但若每次呼叫 API 都要自己組 HTTP 請求，會很痛苦。

為此，OpenAI 團隊寫了一個 python 工具，稱為「Python Client Library」，包裝 HTTP 呼叫。你寫 python，它呼叫 web。

而這就是 `openai` 函式庫。

### 什麼是 `openai` python client library

它是：
- 輕量的 python 工具
- 把你的 python 請求轉成 HTTP 呼叫
- 把 HTTP 回應轉成 python 物件

### 它不是什麼

- 沒有實際執行大型語言模型的程式碼！沒有 GPT 程式碼！只是發 web 請求
- 沒有科學計算程式碼，也沒有 OpenAI 專用邏輯

### 如何使用：

```python
# Create an OpenAI python client for making web calls to OpenAI
openai = OpenAI()

# Make the call
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=[{"role":"user", "content": "what is 2+2?"}])

# Print the result
print(response.choices[0].message.content)
```

### 這在做什麼

當你呼叫 `openai.chat.completions.create()`  
它只是對 `https://api.openai.com/v1/chat/completions` 發 web 請求  
並把回應轉成 python 物件。

就這樣。

若做 [直接 HTTP 呼叫](https://platform.openai.com/docs/guides/text?api-mode=chat&lang=curl)，請見 API 文件  
若用 [Python Client Library](https://platform.openai.com/docs/guides/text?api-mode=chat&lang=python)，請見同一份文件

## 在這個背景下——如何使用其他 LLM？

答案：超級簡單！

其他主要 LLM 都有與 OpenAI 相容的 API endpoint。

OpenAI 幫了大家忙：你們都能用我們的工具把 python 轉成 web 請求。只要把預設的 `https://api.openai/com/v1` 改成你指定的任何 web 位址。

因此即使呼叫非 OpenAI 模型，也能用 OpenAI 工具，例如：

`not_actually_openai = OpenAI(base_url="https://somewhere.completely.different/", api_key="another_providers_key")`

要理解：這段 OpenAI 程式只是對 endpoint 發 HTTP 的 utility。所以我們用 OpenAI 團隊的程式碼，也能呼叫其他公司的模型。

以下是主要供應商的 OpenAI 相容 endpoint，甚至包含本機 Ollama。Ollama 在本機提供 endpoint，且也做成 OpenAI 相容——非常方便。

```python
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROK_BASE_URL = "https://api.x.ai/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
```

## Gemini、DeepSeek、Ollama 與 OpenRouter 範例

### 範例 1：用 Gemini 取代 OpenAI

1. 造訪 Google Studio 建立帳號：https://aistudio.google.com/  
2. 將金鑰以 GOOGLE_API_KEY 加入 `.env`  
3. 再以 GEMINI_API_KEY 加入一次——之後會很有用。

然後：

```python
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
response = gemini.chat.completions.create(model="gemini-2.5-flash-preview-05-20", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```

### 範例 2：用 DeepSeek API 取代 OpenAI（便宜，最低儲值 $2）

1. 造訪 DeepSeek API 建立帳號：https://platform.deepseek.com/  
2. 需先加最低 $2 餘額。  
3. 將金鑰以 DEEPSEEK_API_KEY 加入 `.env`  

然後：

```python
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek = OpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
response = deepseek.chat.completions.create(model="deepseek-chat", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```

### 範例 3：用 Ollama 本機免費取代 OpenAI

Ollama 可在本機跑模型；在你的機器上提供 OpenAI 相容 API。  
Ollama 沒有 API key；沒有第三方握有你的信用卡，因此不需要 key。

1. 若初次使用 Ollama，請至 https://ollama.com 安裝  
2. 在 Cursor Terminal 執行 `ollama run llama3.2` 與 Llama 3.2 聊天  
注意：不要用 llama3.3 或 llama4——這些是為家用電腦設計的巨大模型！會塞滿硬碟。  

然後：

```python
!ollama pull llama3.2

from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```

### 範例 4：用熱門服務 [OpenRouter](https://openrouter.ai)（帳單流程較簡單）取代 OpenAI

OpenRouter 很方便：免費存取許多模型，付費模型也只需小額預付。

1. 在 https://openrouter.ai 註冊
2. 視需要加最低預付餘額
3. 將金鑰以 OPENROUTER_API_KEY 加入 `.env`

然後：

```python
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
response = openrouter.chat.completions.create(model="openai/gpt-4.1-nano", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```


### 在 Agent Framework 中使用不同 API 供應商

Agent Framework 讓切換供應商很容易。課程中可隨時換 LLM。下方有更多說明。OpenAI Agents SDK 見本 notebook 後段。CrewAI 課程會教，但很簡單：用 LiteLLM 期望的完整模型路徑即可。

## API 費用

每次 API 呼叫費用極低——課程多數模型呼叫僅幾美分的一小部分。

但極其重要：

1. 複雜 Agentic 專案可能涉及 20–30 次 LLM 呼叫，費用會累積。請設限並監控用量。

2. Agentic AI 有 agents 迴圈或超出預期處理的風險。請監控 API 用量，預算不要超過你能接受的範圍。部分 API 有「自動儲值」會自動扣款——強烈建議關閉。

3. 只花你能接受的錢。Ollama 是免費替代。DeepSeek、Gemini 2.5 Flash、gpt-4.1-nano 都便宜很多。

記住：這些 LLM 呼叫通常涉及數兆次浮點運算——總得有人付電費！

### Ollama：付費 API 的免費替代（請注意 llama 版本警告）

Ollama 在本機執行，可跑開源模型，並在本機提供與 OpenAI 相容的 API endpoint。

首先至 https://ollama.com 下載 Ollama。

然後在 Cursor Terminal（View >> Terminal）執行以下指令下載模型：

```shell
ollama pull llama3.2
```

警告：不要用 llama3.3 或 llama4——體積大得多，不適合家用電腦。

之後，每當程式碼有：  
`openai = OpenAI()`  
可直接替換為：  
`openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')`  
並把 **gpt-4o-mini** 等模型名換成 **llama3.2**。  

Ollama 不需在 .env 放任何東西；一切都在本機。你不呼叫雲端第三方，沒人握有信用卡，因此不需要 secret key！上面 `api_key='ollama'` 只是因為 OpenAI client library 要求傳入 api_key，Ollama 會忽略該值。

完整範例如下：

```python
# You need to do this one time on your computer
!ollama pull llama3.2

from openai import OpenAI
MODEL = "llama3.2"
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "What is 2 + 2?"}]
)

print(response.choices[0].message.content)
```

在任何 Agent Framework 中使用 Ollama 也需類似修改——可 google 範例，或問我。

### OpenRouter：連接 OpenAI 與其他模型的便利閘道

OpenRouter 是第三方服務，可連接多種 LLM，包含 OpenAI。

以較簡單的帳單流程聞名，對美國以外某些國家可能更容易。

首先造訪：  
https://openrouter.ai/

再看 quickstart：  
https://openrouter.ai/docs/quickstart

將金鑰加入 .env：  
```shell
OPENROUTER_API_KEY=sk-or....
```

之後，每當有這樣的程式碼：  
```python
MODEL = "gpt-4o-mini"
openai = OpenAI()
```

可替換為：

```python
MODEL = "openai/gpt-4o-mini"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "What is 2 + 2?"}]
)

print(response.choices[0].message.content)
```

在 Agent Framework 中使用 OpenRouter 也需類似修改——可 google 範例，或問我。

## OpenAI Agents SDK——專用說明

使用 OpenAI Agents SDK（第 2、6 週）時，使用 OpenAI 自家模型特別簡單。傳入模型名稱即可：

`agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")`

也可替換成任何 OpenAI 相容 API 的供應商，三步驟如下：

```python
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
```

建立 Agent 時傳入此 model 即可。

`agent = Agent(name="Jokester", instructions="You are a joke teller", model=deepseek_model)`

其他 OpenAI 相容 API 也可用相同三步驟：

```python
# extra imports
from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI

# Step 1: specify the base URL endpoints where the provider offers an OpenAI compatible API
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROK_BASE_URL = "https://api.x.ai/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Step 2: Create an AsyncOpenAI object for that endpoint
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Step 3: Create a model object to provide when creating an Agent
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.5-flash", openai_client=gemini_client)
grok_3_model = OpenAIChatCompletionsModel(model="grok-3-mini-beta", openai_client=openrouter_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)
grok_3_via_openrouter_model = OpenAIChatCompletionsModel(model="x-ai/grok-3-mini-beta", openai_client=openrouter_client)
llama_3_2_local_model = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama_client)
```

### 在 OpenAI Agents SDK 中使用 Azure

說明見：  
https://techcommunity.microsoft.com/blog/azure-ai-services-blog/use-azure-openai-and-apim-with-the-openai-agents-sdk/4392537

例如：
```python
from openai import AsyncAzureOpenAI
from agents import set_default_openai_client
from dotenv import load_dotenv
import os
 
# Load environment variables
load_dotenv()
 
# Create OpenAI client using Azure OpenAI
openai_client = AsyncAzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT")
)
 
# Set the default OpenAI client for the Agents SDK
set_default_openai_client(openai_client)
```

## CrewAI 設定

以下是 Crew 的 LLM 連線文件與各模型名稱。學員 Sadan S. 指出（感謝！），Google 需用環境變數 `GEMINI_API_KEY` 而非 `GOOGLE_API_KEY`：

https://docs.crewai.com/concepts/llms

教學與更多資訊：

https://docs.crewai.com/how-to/llm-connections

## LangGraph 設定

在 LangGraph 中使用 Ollama（其他模型類似）：  
https://python.langchain.com/docs/integrations/chat/ollama/#installation

先加套件：  
`uv add langchain-ollama`

在 lab 中替換：   
```python
from langchain_ollama import ChatOllama
# llm = ChatOpenAI(model="gpt-4o-mini")
llm = ChatOllama(model="gemma3:4b")
```

並事先執行 `!ollama pull gemma3:4b`（或你選的模型）。

感謝 Miroslav P. 補充，感謝 Arvin F. 提問！

## LangGraph 與其他模型

沿用上述做法，模型列表見：  
https://python.langchain.com/docs/integrations/chat/


## AutoGen 與其他模型

Miroslav P. 的另一貢獻（感謝！）——在 AutoGen 中用 Ollama + 本機模型，並有 gemma3 表現良好的範例。

```python
# model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
 
from autogen_ext.models.ollama import OllamaChatCompletionClient
 
model_client = OllamaChatCompletionClient(
    model="gemma3:4b",
    model_info={
        "vision": True,
        "function_calling": False,
        "json_output": True,
        "family": "unknown",
    },
)
```

## 值得記住

1. 若用 Ollama 本機跑模型，較小模型可能在進階專案中吃力。需實驗不同大小與能力，可能需要耐心。我預期 llama3.2 對部分專案太難。替代：openrouter.ai 的免費模型，或幾乎免費的便宜模型——如 DeepSeek。

2. Chat 模型常比 Reasoning 模型表現更好，因為 Reasoning 模型可能對某些作業「想太多」。務必實驗。更大不一定更好……

3. 容易混淆：有兩家名字很像的供應商！  
- Grok 是 Elon Musk 的 X 的 LLM
- Groq 是快速推理開源模型的平台

學員提醒我：「Groq」這個名字先出現！
